# ETL da Olist: do CSV bruto ao modelo estrela

Projeto Integrador IV (Banco de Dados) - Etapa 2 | Mineração de Dados

Este notebook executa as **transformações de dados** do projeto: lê os CSVs da Olist (Kaggle),
filtra e trata os pedidos entregues, deriva métricas de receita e entrega e organiza tudo em um
**modelo estrela** (1 tabela-fato + 6 dimensões), salvo em arquivos `parquet` que alimentam o
dashboard em Streamlit.

O mesmo código está em `etl/transformacao.py` (para rodar via linha de comando).

| Etapa | O que faz |
|---|---|
| 1. Ingestão | baixa o dataset do Kaggle e lê 6 tabelas |
| 2. Limpeza | mantém só pedidos entregues com data de entrega |
| 3. Dimensões | tempo, cliente, produto, vendedor, pagamento e entrega |
| 4. Fato | grão de item de pedido, com receita, prazo e dias de atraso |
| 5. Qualidade | contagens, outliers e integridade das chaves |


## 1. Ingestão dos dados (Kaggle)

Para baixar direto no Colab é preciso o token de API do Kaggle (`kaggle.json`):
**kaggle.com/settings > API > Create New Token**. Rode a célula abaixo e selecione o arquivo
quando o botão de upload aparecer.

In [3]:
from google.colab import files
uploaded = files.upload()  # selecione o kaggle.json

Saving kaggle.json to kaggle.json


In [4]:
!pip install -q kaggle pyarrow
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -o -q brazilian-ecommerce.zip -d olist_data
!ls olist_data

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:00<00:00, 147MB/s]

olist_customers_dataset.csv	  olist_orders_dataset.csv
olist_geolocation_dataset.csv	  olist_products_dataset.csv
olist_order_items_dataset.csv	  olist_sellers_dataset.csv
olist_order_payments_dataset.csv  product_category_name_translation.csv
olist_order_reviews_dataset.csv


## 2. Funções de transformação

Cada função abaixo cuida de uma parte do pipeline. A última (`executar`) encadeia todas.

In [5]:
import json
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

### Constantes (rótulos em português e faixas de classificação)

In [6]:
DIAS_SEMANA = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
MESES = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro",
]
TIPOS_PAGAMENTO = {
    "credit_card": "Cartão de crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de débito",
    "not_defined": "Não definido",
}
FAIXAS_PARCELAS = ["À vista (1x)", "2 a 5x", "6 a 10x", "Mais de 10x"]
FAIXAS_PRAZO = ["Até 7 dias", "8 a 14 dias", "15 a 30 dias", "Mais de 30 dias"]
SITUACOES_ENTREGA = ["No prazo", "Atrasado"]
COLUNAS_DATA = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

### Ingestão: leitura dos 6 CSVs

Le os seis CSVs da Olist necessarios para o modelo estrela.

In [7]:
def carregar_tabelas(origem):
    """Le os seis CSVs da Olist necessarios para o modelo estrela."""
    origem = Path(origem)
    return {
        "orders": pd.read_csv(origem / "olist_orders_dataset.csv", parse_dates=COLUNAS_DATA),
        "order_items": pd.read_csv(origem / "olist_order_items_dataset.csv"),
        "customers": pd.read_csv(origem / "olist_customers_dataset.csv"),
        "products": pd.read_csv(origem / "olist_products_dataset.csv"),
        "sellers": pd.read_csv(origem / "olist_sellers_dataset.csv"),
        "payments": pd.read_csv(origem / "olist_order_payments_dataset.csv"),
    }

### Limpeza: pedidos entregues com data de entrega

Mantem so pedidos entregues e com data de entrega preenchida.

In [8]:
def filtrar_pedidos_entregues(orders):
    """Mantem so pedidos entregues e com data de entrega preenchida."""
    entregues = orders[orders["order_status"] == "delivered"]
    sem_data = entregues["order_delivered_customer_date"].isna()
    validos = entregues[~sem_data].copy()
    log = {
        "pedidos_total": int(len(orders)),
        "pedidos_entregues": int(len(entregues)),
        "entregues_sem_data_de_entrega_removidos": int(sem_data.sum()),
        "pedidos_validos": int(len(validos)),
    }
    return validos, log

### Dimensão tempo

Calendario continuo, do primeiro ao ultimo dia de compra.

In [9]:
def construir_dim_tempo(datas):
    """Calendario continuo, do primeiro ao ultimo dia de compra."""
    dias = pd.date_range(datas.min().normalize(), datas.max().normalize(), freq="D")
    dim = pd.DataFrame({"data": dias})
    dim["id_tempo"] = dim["data"].dt.strftime("%Y%m%d").astype(int)
    dim["ano"] = dim["data"].dt.year.astype(int)
    dim["mes"] = dim["data"].dt.month.astype(int)
    dim["nome_mes"] = dim["mes"].map(lambda m: MESES[m - 1])
    dim["ano_mes"] = dim["data"].dt.strftime("%Y-%m")
    dim["trimestre"] = dim["data"].dt.quarter.astype(int)
    dim["num_dia_semana"] = dim["data"].dt.dayofweek.astype(int)
    dim["dia_semana"] = dim["num_dia_semana"].map(lambda i: DIAS_SEMANA[i])
    return dim

### Dimensão cliente



In [10]:
def construir_dim_cliente(customers):
    dim = customers.rename(
        columns={
            "customer_id": "id_cliente",
            "customer_unique_id": "id_cliente_unico",
            "customer_zip_code_prefix": "cep_prefixo_cliente",
            "customer_city": "cidade_cliente",
            "customer_state": "uf_cliente",
        }
    )[["id_cliente", "id_cliente_unico", "cep_prefixo_cliente", "cidade_cliente", "uf_cliente"]]
    dim = dim.drop_duplicates("id_cliente").copy()
    dim["cidade_cliente"] = dim["cidade_cliente"].str.title()
    return dim

### Dimensão produto



In [11]:
def construir_dim_produto(products):
    dim = products.rename(
        columns={
            "product_id": "id_produto",
            "product_category_name": "categoria",
            "product_weight_g": "peso_g",
            "product_length_cm": "comprimento_cm",
            "product_height_cm": "altura_cm",
            "product_width_cm": "largura_cm",
            "product_photos_qty": "qtd_fotos",
        }
    )[["id_produto", "categoria", "peso_g", "comprimento_cm", "altura_cm", "largura_cm", "qtd_fotos"]]
    dim = dim.drop_duplicates("id_produto").copy()
    dim["categoria"] = dim["categoria"].fillna("sem_categoria")
    return dim

### Dimensão vendedor



In [12]:
def construir_dim_vendedor(sellers):
    dim = sellers.rename(
        columns={
            "seller_id": "id_vendedor",
            "seller_zip_code_prefix": "cep_prefixo_vendedor",
            "seller_city": "cidade_vendedor",
            "seller_state": "uf_vendedor",
        }
    )[["id_vendedor", "cep_prefixo_vendedor", "cidade_vendedor", "uf_vendedor"]]
    dim = dim.drop_duplicates("id_vendedor").copy()
    dim["cidade_vendedor"] = dim["cidade_vendedor"].str.title()
    return dim

### Pagamento principal por pedido

Um pedido pode ter varios pagamentos; fica o de maior valor (principal).

Assim o pagamento nao multiplica a receita ao ser ligado aos itens.

In [13]:
def preparar_pagamento_por_pedido(payments):
    """Um pedido pode ter varios pagamentos; fica o de maior valor (principal).

    Assim o pagamento nao multiplica a receita ao ser ligado aos itens.
    """
    ordenado = payments.sort_values(
        ["order_id", "payment_value", "payment_sequential"], ascending=[True, False, True]
    )
    principal = ordenado.drop_duplicates("order_id", keep="first").copy()
    principal["tipo_pagamento"] = principal["payment_type"].map(TIPOS_PAGAMENTO).fillna("Não definido")
    principal["faixa_parcelas"] = pd.cut(
        principal["payment_installments"],
        bins=[-np.inf, 1, 5, 10, np.inf],
        labels=FAIXAS_PARCELAS,
    ).astype(str)
    return principal[["order_id", "tipo_pagamento", "faixa_parcelas"]]

### Dimensão pagamento

Dimensao pequena: combinacoes de tipo de pagamento e faixa de parcelas.

In [14]:
def construir_dim_pagamento(pagamento_por_pedido):
    """Dimensao pequena: combinacoes de tipo de pagamento e faixa de parcelas."""
    combos = (
        pagamento_por_pedido[["tipo_pagamento", "faixa_parcelas"]]
        .drop_duplicates()
        .sort_values(["tipo_pagamento", "faixa_parcelas"])
        .reset_index(drop=True)
    )
    combos.insert(0, "id_pagamento", np.arange(1, len(combos) + 1))
    nao_informado = pd.DataFrame(
        {"id_pagamento": [0], "tipo_pagamento": ["Não informado"], "faixa_parcelas": ["Não informado"]}
    )
    return pd.concat([nao_informado, combos], ignore_index=True)

### Dimensão entrega

Dimensao fixa: situacao da entrega x faixa de prazo.

In [15]:
def construir_dim_entrega():
    """Dimensao fixa: situacao da entrega x faixa de prazo."""
    linhas = [(s, f) for s in SITUACOES_ENTREGA for f in FAIXAS_PRAZO]
    dim = pd.DataFrame(linhas, columns=["situacao_entrega", "faixa_prazo"])
    dim.insert(0, "id_entrega", np.arange(1, len(dim) + 1))
    return dim

### Tabela-fato (grão: item de pedido)

Tabela-fato no grao de item de pedido, com medidas de receita e entrega.

In [16]:
def construir_fato(order_items, pedidos, pagamento_por_pedido, dim_pagamento, dim_entrega):
    """Tabela-fato no grao de item de pedido, com medidas de receita e entrega."""
    cols_pedido = [
        "order_id", "customer_id", "order_purchase_timestamp",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ]
    fato = order_items.merge(pedidos[cols_pedido], on="order_id", how="inner")

    pagamentos = pagamento_por_pedido.merge(
        dim_pagamento, on=["tipo_pagamento", "faixa_parcelas"], how="left"
    )[["order_id", "id_pagamento"]]
    fato = fato.merge(pagamentos, on="order_id", how="left")
    fato["id_pagamento"] = fato["id_pagamento"].fillna(0).astype(int)

    compra = fato["order_purchase_timestamp"]
    entrega = fato["order_delivered_customer_date"]
    estimada = fato["order_estimated_delivery_date"]

    fato["id_tempo"] = compra.dt.strftime("%Y%m%d").astype(int)
    fato["hora_compra"] = compra.dt.hour.astype(int)
    fato["prazo_entrega_dias"] = (entrega - compra).dt.days.astype(int)
    fato["prazo_estimado_dias"] = (estimada - compra).dt.days.astype(int)
    fato["dias_atraso"] = (entrega.dt.normalize() - estimada).dt.days.clip(lower=0).astype(int)

    situacao = np.where(fato["dias_atraso"] > 0, "Atrasado", "No prazo")
    faixa = pd.cut(
        fato["prazo_entrega_dias"],
        bins=[-np.inf, 7, 14, 30, np.inf],
        labels=FAIXAS_PRAZO,
    ).astype(str)
    chave = pd.DataFrame({"situacao_entrega": situacao, "faixa_prazo": faixa})
    chave = chave.merge(dim_entrega, on=["situacao_entrega", "faixa_prazo"], how="left")
    fato["id_entrega"] = chave["id_entrega"].to_numpy()

    fato["preco"] = fato["price"]
    fato["frete"] = fato["freight_value"]
    fato["receita"] = fato["preco"] + fato["frete"]

    limite = float(fato["prazo_entrega_dias"].quantile(0.98))
    fato["prazo_outlier"] = fato["prazo_entrega_dias"] > limite

    fato = fato.rename(
        columns={
            "order_id": "id_pedido",
            "order_item_id": "id_item",
            "customer_id": "id_cliente",
            "product_id": "id_produto",
            "seller_id": "id_vendedor",
        }
    )
    colunas = [
        "id_pedido", "id_item", "id_tempo", "hora_compra", "id_cliente", "id_produto",
        "id_vendedor", "id_pagamento", "id_entrega", "preco", "frete", "receita",
        "prazo_entrega_dias", "prazo_estimado_dias", "dias_atraso", "prazo_outlier",
    ]
    return fato[colunas].copy(), limite

### Qualidade: integridade das chaves

Conta chaves da fato sem correspondente na dimensao (esperado: zero).

In [17]:
def checar_integridade(fato, dims):
    """Conta chaves da fato sem correspondente na dimensao (esperado: zero)."""
    chaves = {
        "id_tempo": "dim_tempo", "id_cliente": "dim_cliente", "id_produto": "dim_produto",
        "id_vendedor": "dim_vendedor", "id_pagamento": "dim_pagamento", "id_entrega": "dim_entrega",
    }
    orfas = {}
    for chave, nome_dim in chaves.items():
        orfas[chave] = int((~fato[chave].isin(dims[nome_dim][chave])).sum())
    return orfas

### Gravação dos arquivos parquet



In [18]:
def salvar(tabelas, destino):
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    for nome, df in tabelas.items():
        df.to_parquet(destino / f"{nome}.parquet", index=False)

### Pipeline completo

Roda o pipeline completo e devolve o relatorio de qualidade.

In [19]:
def executar(origem, destino):
    """Roda o pipeline completo e devolve o relatorio de qualidade."""
    bruto = carregar_tabelas(origem)

    pedidos, log = filtrar_pedidos_entregues(bruto["orders"])

    dim_tempo = construir_dim_tempo(pedidos["order_purchase_timestamp"])
    dim_cliente = construir_dim_cliente(bruto["customers"])
    dim_produto = construir_dim_produto(bruto["products"])
    dim_vendedor = construir_dim_vendedor(bruto["sellers"])
    pagamento_por_pedido = preparar_pagamento_por_pedido(bruto["payments"])
    dim_pagamento = construir_dim_pagamento(pagamento_por_pedido)
    dim_entrega = construir_dim_entrega()

    fato, limite_outlier = construir_fato(
        bruto["order_items"], pedidos, pagamento_por_pedido, dim_pagamento, dim_entrega
    )

    tabelas = {
        "fato_pedidos": fato,
        "dim_tempo": dim_tempo,
        "dim_cliente": dim_cliente,
        "dim_produto": dim_produto,
        "dim_vendedor": dim_vendedor,
        "dim_pagamento": dim_pagamento,
        "dim_entrega": dim_entrega,
    }
    salvar(tabelas, destino)

    pedidos_na_fato = fato["id_pedido"].nunique()
    relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        **log,
        "itens_na_fato": int(len(fato)),
        "pedidos_na_fato": int(pedidos_na_fato),
        "pedidos_sem_pagamento": int((fato.drop_duplicates("id_pedido")["id_pagamento"] == 0).sum()),
        "produtos_sem_categoria": int((dim_produto["categoria"] == "sem_categoria").sum()),
        "limite_outlier_prazo_dias_p98": limite_outlier,
        "itens_marcados_outlier_prazo": int(fato["prazo_outlier"].sum()),
        "percentual_pedidos_atrasados": round(
            float((fato.drop_duplicates("id_pedido")["dias_atraso"] > 0).mean() * 100), 2
        ),
        "chaves_orfas": checar_integridade(fato, tabelas),
        "linhas_por_tabela": {nome: int(len(df)) for nome, df in tabelas.items()},
    }
    with open(Path(destino) / "qualidade_etl.json", "w", encoding="utf-8") as f:
        json.dump(relatorio, f, ensure_ascii=False, indent=2)
    return relatorio

## 3. Executando o pipeline

Gera os arquivos do modelo estrela na pasta `data/` e imprime o relatório de qualidade.

In [20]:
import json
relatorio = executar("olist_data", "data")
print(json.dumps(relatorio, ensure_ascii=False, indent=2))

{
  "gerado_em": "2026-11-23 11:55:32",
  "pedidos_total": 99441,
  "pedidos_entregues": 96478,
  "entregues_sem_data_de_entrega_removidos": 8,
  "pedidos_validos": 96470,
  "itens_na_fato": 110189,
  "pedidos_na_fato": 96470,
  "pedidos_sem_pagamento": 1,
  "produtos_sem_categoria": 610,
  "limite_outlier_prazo_dias_p98": 38.0,
  "itens_marcados_outlier_prazo": 2141,
  "percentual_pedidos_atrasados": 6.77,
  "chaves_orfas": {
    "id_tempo": 0,
    "id_cliente": 0,
    "id_produto": 0,
    "id_vendedor": 0,
    "id_pagamento": 0,
    "id_entrega": 0
  },
  "linhas_por_tabela": {
    "fato_pedidos": 110189,
    "dim_tempo": 714,
    "dim_cliente": 99441,
    "dim_produto": 32951,
    "dim_vendedor": 3095,
    "dim_pagamento": 9,
    "dim_entrega": 8
  }
}


### Conferindo o resultado

In [21]:
fato = pd.read_parquet("data/fato_pedidos.parquet")
print(fato.shape)
fato.head()

(110189, 16)


,id_pedido,id_item,id_tempo,hora_compra,id_cliente,id_produto,id_vendedor,id_pagamento,id_entrega,preco,frete,receita,prazo_entrega_dias,prazo_estimado_dias,dias_atraso,prazo_outlier
0,00010242fe8c5a6d1ba2dd792cb16214,1,20170913,8,3ce436f183e68e07877b285a838db11a,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2,1,58.90,13.29,72.19,7,15,0,False
1,00018f77f2f0320c557190d7a144bdd3,1,20170426,10,f6dd3ec061db4e3987629fe6b26e5cce,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2,3,239.90,19.93,259.83,16,18,0,False
2,000229ec398224ef6ca0657da4fc703e,1,20180114,14,6489ae5e4333f3693df5ad4372dab6d3,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2,1,199.00,17.87,216.87,7,21,0,False
3,00024acbcdf0a6daa1e931b038114c75,1,20180808,10,d4eb9395c8c0431ee92fce09860c5a06,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2,1,12.99,12.79,25.78,6,11,0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,20170204,13,58dbd0b2d70206bf40e62cd34e84d795,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2,3,199.90,18.14,218.04,25,40,0,False


In [22]:
fato[["receita", "prazo_entrega_dias", "dias_atraso"]].describe().round(2)

,receita,prazo_entrega_dias,dias_atraso
count,110189.00,110189.00,110189.00
mean,139.93,12.01,0.69
std,189.32,9.45,4.54
min,6.08,0.00,0.00
25%,55.18,6.00,0.00
50%,92.12,10.00,0.00
75%,157.47,15.00,0.00
max,6929.31,209.00,188.00


In [23]:
# % de pedidos entregues com atraso (por pedido, não por item)
pedidos = fato.drop_duplicates("id_pedido")
print(f"Pedidos atrasados: {(pedidos['dias_atraso'] > 0).mean() * 100:.1f}%")
print(f"Prazo médio de entrega: {pedidos['prazo_entrega_dias'].mean():.1f} dias")

Pedidos atrasados: 6.8%
Prazo médio de entrega: 12.1 dias


## 4. Baixando os arquivos para o repositório

O comando abaixo compacta a pasta `data/` em um zip e baixa para o seu computador.
Descompacte e envie os arquivos `.parquet` e `qualidade_etl.json` para a pasta `data/` do repositório no GitHub.

In [24]:
import shutil
shutil.make_archive("dados_modelo_estrela", "zip", "data")
files.download("dados_modelo_estrela.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import pandas as pd

# 1) Valores do modelo estrela (o que o dashboard usa)
fato = pd.read_parquet("data/fato_pedidos.parquet")
prod = pd.read_parquet("data/dim_produto.parquet")[["id_produto", "categoria"]]
vend = pd.read_parquet("data/dim_vendedor.parquet")[["id_vendedor", "uf_vendedor"]]
df = fato.merge(prod, on="id_produto").merge(vend, on="id_vendedor")
cat_dw = df.groupby("categoria")["receita"].sum().sort_values(ascending=False)
uf_dw = df.groupby("uf_vendedor")["receita"].sum().sort_values(ascending=False)

# 2) Os mesmos totais direto dos CSVs originais
p = "olist_data/"
orders = pd.read_csv(p + "olist_orders_dataset.csv")
itens = pd.read_csv(p + "olist_order_items_dataset.csv")
produtos = pd.read_csv(p + "olist_products_dataset.csv")
vendedores = pd.read_csv(p + "olist_sellers_dataset.csv")

ok = orders[(orders.order_status == "delivered")
            & orders.order_delivered_customer_date.notna()][["order_id"]]
raw = (itens.merge(ok, on="order_id")
            .merge(produtos[["product_id", "product_category_name"]], on="product_id", how="left")
            .merge(vendedores[["seller_id", "seller_state"]], on="seller_id", how="left"))
raw["receita"] = raw["price"] + raw["freight_value"]
raw["product_category_name"] = raw["product_category_name"].fillna("sem_categoria")

cat_raw = raw.groupby("product_category_name")["receita"].sum().sort_values(ascending=False)
uf_raw = raw.groupby("seller_state")["receita"].sum().sort_values(ascending=False)

# 3) Diferença máxima (esperado: perto de zero)
print("Maior diferença em categorias (R$):", cat_dw.sub(cat_raw, fill_value=0).abs().max())
print("Maior diferença em UF do vendedor (R$):", uf_dw.sub(uf_raw, fill_value=0).abs().max())
print("Categorias no DW:", len(cat_dw), "| nos CSVs:", len(cat_raw))
print("UF no DW:", len(uf_dw), "| nos CSVs:", len(uf_raw))
print(cat_dw.head(10))
print(uf_dw.head(10))

Maior diferença em categorias (R$): 0.0
Maior diferença em UF do vendedor (R$): 0.0
Categorias no DW: 74 | nos CSVs: 74
UF no DW: 22 | nos CSVs: 22
categoria
beleza_saude              1412089.53
relogios_presentes        1264016.98
cama_mesa_banho           1225209.26
esporte_lazer             1118062.91
informatica_acessorios    1032603.65
moveis_decoracao           880329.92
utilidades_domesticas      758392.25
cool_stuff                 691680.89
automotivo                 669319.92
ferramentas_jardim         567145.68
Name: receita, dtype: float64
uf_vendedor
SP    9955994.13
PR    1424161.25
MG    1184111.40
RJ     911918.67
SC     717739.37
RS     429749.09
BA     297146.93
DF     112892.78
PE     103486.10
GO      77089.48
Name: receita, dtype: float64
